In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!ls /content/drive/MyDrive


'2 3 * qs'	    coupled_ms.docx   isl_dataset_npy	  landmarks_npy
'Colab Notebooks'   isl_dataset       isl_dataset_split   PracticeSet5.pdf


In [13]:
import os, numpy as np

DATASET_PATH = "/content/drive/MyDrive/isl_dataset_split"

def extract_landmarks(dataset_path, split):
    # Example placeholder: replace with actual landmark extraction logic
    X, y = [], []
    split_path = os.path.join(dataset_path, split)
    for label in os.listdir(split_path):
        folder = os.path.join(split_path, label)
        if os.path.isdir(folder):
            for f in os.listdir(folder):
                # TODO: replace with actual landmark extraction
                X.append(np.random.rand(50, 3))  # fake landmark vector
                y.append(label)
    return np.array(X), np.array(y)

for split in ["train", "val", "test"]:
    X, y = extract_landmarks(DATASET_PATH, split)
    np.save(f"X_{split}.npy", X)
    np.save(f"y_{split}.npy", y)

In [14]:
!mkdir -p /content/drive/MyDrive/isl_dataset_npy
!mv X_*.npy y_*.npy /content/drive/MyDrive/isl_dataset_npy/


In [15]:
import numpy as np

X_train = np.load("/content/drive/MyDrive/isl_dataset_npy/X_train.npy")
y_train = np.load("/content/drive/MyDrive/isl_dataset_npy/y_train.npy")
X_val   = np.load("/content/drive/MyDrive/isl_dataset_npy/X_val.npy")
y_val   = np.load("/content/drive/MyDrive/isl_dataset_npy/y_val.npy")
X_test  = np.load("/content/drive/MyDrive/isl_dataset_npy/X_test.npy")
y_test  = np.load("/content/drive/MyDrive/isl_dataset_npy/y_test.npy")

In [16]:
!ls /content/drive/MyDrive

'2 3 * qs'	    coupled_ms.docx   isl_dataset_npy	  landmarks_npy
'Colab Notebooks'   isl_dataset       isl_dataset_split   PracticeSet5.pdf


In [17]:
!ls /content/drive/MyDrive/isl_dataset_npy

X_test.npy  X_train.npy  X_val.npy  y_test.npy	y_train.npy  y_val.npy


In [18]:
!ls /content/drive/MyDrive/isl_dataset_npy
!ls /content/drive/MyDrive/landmarks_npy

X_test.npy  X_train.npy  X_val.npy  y_test.npy	y_train.npy  y_val.npy
X_test.npy  X_train.npy  X_val.npy  y_test.npy	y_train.npy  y_val.npy


In [19]:
!ls /content/drive/MyDrive/isl_dataset_npy
!ls /content/drive/MyDrive/landmarks_npy

X_test.npy  X_train.npy  X_val.npy  y_test.npy	y_train.npy  y_val.npy
X_test.npy  X_train.npy  X_val.npy  y_test.npy	y_train.npy  y_val.npy


In [20]:
X_train = np.load("/content/drive/MyDrive/isl_dataset_npy/X_train.npy")

In [21]:
import numpy as np

X_train = np.load("/content/drive/MyDrive/isl_dataset_npy/X_train.npy")
y_train = np.load("/content/drive/MyDrive/isl_dataset_npy/y_train.npy")
X_val   = np.load("/content/drive/MyDrive/isl_dataset_npy/X_val.npy")
y_val   = np.load("/content/drive/MyDrive/isl_dataset_npy/y_val.npy")
X_test  = np.load("/content/drive/MyDrive/isl_dataset_npy/X_test.npy")
y_test  = np.load("/content/drive/MyDrive/isl_dataset_npy/y_test.npy")

In [22]:
import tensorflow as tf
from tensorflow.keras import layers, models

inputs = tf.keras.Input(shape=(X_train.shape[1], X_train.shape[2]))

x = layers.Conv1D(64, 3, activation='relu')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(128, 3, activation='relu')(x)
x = layers.GlobalAveragePooling1D()(x)

# Attention mechanism
attention_probs = layers.Dense(x.shape[-1], activation='softmax')(x)
x = layers.multiply([x, attention_probs])

outputs = layers.Dense(len(set(y_train)), activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [24]:
from sklearn.preprocessing import LabelEncoder

# Encode labels (convert strings → integers)
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_val   = encoder.transform(y_val)
y_test  = encoder.transform(y_test)

print("Classes:", encoder.classes_)
print("Sample encoded labels:", y_train[:10])

Classes: ['0' '1' '2' '3' '4' '5' '6' '7' '8' '9' 'A' 'B' 'C' 'D' 'E' 'F' 'G' 'H'
 'I']
Sample encoded labels: [5 5 5 5 5 5 5 5 5 5]


In [25]:
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,       # start small for speed
                    batch_size=64)   # larger batch speeds up training

Epoch 1/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.0521 - loss: 2.9374 - val_accuracy: 0.0539 - val_loss: 2.9355
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0535 - loss: 2.9333 - val_accuracy: 0.0582 - val_loss: 2.9350
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0561 - loss: 2.9327 - val_accuracy: 0.0506 - val_loss: 2.9358
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0527 - loss: 2.9330 - val_accuracy: 0.0560 - val_loss: 2.9358
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0532 - loss: 2.9329 - val_accuracy: 0.0468 - val_loss: 2.9363
Epoch 6/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0534 - loss: 2.9330 - val_accuracy: 0.0468 - val_loss: 2.9355
Epoch 7/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0516 - loss: 2.9327 - val_accuracy: 0.0468 - val_loss: 2.9362
Epoch 8/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0543 - loss: 2.9325 - val_accuracy: 0

In [26]:
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,       # start small for speed
                    batch_size=64)   # larger batch speeds up training

Epoch 1/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0530 - loss: 2.9324 - val_accuracy: 0.0468 - val_loss: 2.9366
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0543 - loss: 2.9327 - val_accuracy: 0.0468 - val_loss: 2.9353
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0547 - loss: 2.9326 - val_accuracy: 0.0468 - val_loss: 2.9366
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0502 - loss: 2.9325 - val_accuracy: 0.0468 - val_loss: 2.9357
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0555 - loss: 2.9325 - val_accuracy: 0.0468 - val_loss: 2.9353
Epoch 6/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0503 - loss: 2.9325 - val_accuracy: 0.0501 - val_loss: 2.9357
Epoch 7/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0537 - loss: 2.9326 - val_accuracy: 0.0517 - val_loss: 2.9351
Epoch 8/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0543 - loss: 2.9323 - val_accuracy: 0.

In [27]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test accuracy:", test_acc)

58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0538 - loss: 2.9270
Test accuracy: 0.05380434915423393


In [28]:
model.save("/content/drive/MyDrive/isl_model_cnn_attention.h5")

In [29]:
from tensorflow.keras.models import load_model
model = load_model("/content/drive/MyDrive/isl_model_cnn_attention.h5")